# BiLSTM Adversarial Examples: Test

This notebook generates gradient-guided character edits against the trained BiLSTM, validates them, and evaluates the model on the original test set plus non-duplicate adversarial rows.

In [14]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, "../src")
import generate_bilstm_adversarial as attack_impl

## Setup

The model uses the project character vocabulary, a 256-character input limit, and a maximum edit budget of 3.

In [15]:
SPLIT = "test"
N = None  # set to a small number such as 10 for a smoke test
BATCH_SIZE = 64
TOP_POSITIONS = 3
SEED = 42

CHECKPOINT = Path("../models/bilstm_model1.pt")
OUTPUT = Path("../data/bilstm_adv_test.csv")
PARTIAL = Path("../data/bilstm_adv_test_partial.csv")
RESULTS_DIR = Path("../results")

config, vocab = attack_impl.load_config()
MAX_LEN = config["max_len"]
K = config["budget"]
INDEX_TO_STRING = vocab["index_to_string"]
CHAR_TO_IDX = {character: index for index, character in enumerate(INDEX_TO_STRING)}
PAD_IDX = vocab["pad_idx"]
UNK_IDX = vocab["unk_idx"]
DEVICE = "cuda" if attack_impl.torch.cuda.is_available() else "mps" if attack_impl.torch.backends.mps.is_available() else "cpu"
torch = attack_impl.torch
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", DEVICE, "| split:", SPLIT, "| max_len:", MAX_LEN, "| k:", K)

device: cuda | split: test | max_len: 256 | k: 3


## Load Data and Model

In [16]:
splits = attack_impl.load_splits()
model = attack_impl.load_model(CHECKPOINT, len(CHAR_TO_IDX), PAD_IDX, DEVICE)
embedding = model.embedding.weight
valid_embedding_ids = torch.tensor([index for character, index in CHAR_TO_IDX.items() if character not in {"<PAD>", "<UNK>"}], dtype=torch.long, device=DEVICE)
p_phish = attack_impl.make_predictor(model, CHAR_TO_IDX, UNK_IDX, PAD_IDX, MAX_LEN, DEVICE)
print("checkpoint:", CHECKPOINT, "| vocabulary:", len(CHAR_TO_IDX))

checkpoint: ../models/bilstm_model1.pt | vocabulary: 67


## Select Attack Targets

Only phishing URLs that the clean BiLSTM currently detects are attacked. The original phishing probability is kept as `p_clean`.

In [17]:
phishing = attack_impl.drop_invalid_originals(splits[SPLIT][splits[SPLIT].label == 0]).copy()
p_all = p_phish(phishing.URL.tolist())
targets = phishing[p_all >= 0.5].copy()
targets["p_clean"] = p_all[p_all >= 0.5]
if N is not None:
    targets = targets.sample(min(N, len(targets)), random_state=SEED)
print(f"phishing: {len(phishing)} | missed: {(p_all < 0.5).sum()} | to attack: {len(targets)}")

AttributeError: module 'generate_bilstm_adversarial' has no attribute 'drop_invalid_originals'

## Attack

The BiLSTM is character-based, so each gradient position already corresponds to one URL character. The shared attack function ranks positions using gradients through the character embedding matrix, then tries valid substitutions, insertions, deletions, and appends.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
PARTIAL.parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if PARTIAL.exists():
    completed = pd.read_csv(PARTIAL, keep_default_na=False)
    print(f"Resuming: {len(completed)} URLs already generated")
else:
    completed = pd.DataFrame(columns=["orig_URL", "URL"])
    completed.to_csv(PARTIAL, index=False)

todo = targets[~targets.URL.isin(completed.orig_URL)]
start_time = time.time()
for start in tqdm(range(0, len(todo), BATCH_SIZE), desc=f"attack {SPLIT}"):
    chunk = todo.iloc[start:start + BATCH_SIZE]
    adversarial = attack_impl.attack_batch(
        chunk.URL.tolist(), chunk.p_clean.tolist(), model, embedding,
        valid_embedding_ids, CHAR_TO_IDX, UNK_IDX, PAD_IDX, MAX_LEN, K,
        TOP_POSITIONS, DEVICE, p_phish
    )
    pd.DataFrame({"orig_URL": chunk.URL.values, "URL": adversarial}).to_csv(
        PARTIAL, mode="a", header=False, index=False
    )

attack_source = PARTIAL
print("remaining URLs:", len(todo))
print("seconds per URL:", round((time.time() - start_time) / max(1, len(todo)), 4))

attack test:   0%|          | 0/124 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Validate and Save

In [ ]:
attack_results = pd.read_csv(PARTIAL, keep_default_na=False)
if attack_results["orig_URL"].duplicated().any():
    raise ValueError("Partial attack results contain duplicate original URLs.")

attack_map = dict(zip(attack_results["orig_URL"], attack_results["URL"]))
missing = targets.loc[~targets.URL.isin(attack_map), "URL"]
if not missing.empty:
    raise RuntimeError(f"Missing attack results for {len(missing)} target URLs; rerun the attack cell.")

adv_df = pd.DataFrame({"orig_URL": targets.URL.values, "URL": [attack_map[url] for url in targets.URL], "label": 0, "p_clean": targets.p_clean.values})
adv_df["p_adv"] = p_phish(adv_df.URL.tolist())
adv_df["n_edits"] = [attack_impl.edit_distance(original, edited) for original, edited in zip(adv_df.orig_URL, adv_df.URL)]
adv_df["flipped"] = adv_df.p_adv < 0.5
reasons = pd.Series([attack_impl.invalid_reason(original, edited, K) for original, edited in zip(adv_df.orig_URL, adv_df.URL)])
print(reasons.replace("", "valid").value_counts())
assert (reasons == "").all()
adv_df["duplicate"] = (adv_df.n_edits > 0) & (adv_df.URL.isin(splits[SPLIT].URL) | adv_df.URL.duplicated())
summary = {"model": "bilstm", "split": SPLIT, "phishing": len(phishing), "missed_by_model": int((p_all < 0.5).sum()), **attack_impl.attack_summary(adv_df, K), "n_edits": {int(count): int(number) for count, number in adv_df.n_edits.value_counts().sort_index().items()}}
adv_df.to_csv(OUTPUT, index=False)
with (RESULTS_DIR / "bilstm_adv_test_summary.json").open("w") as file:
    json.dump(summary, file, indent=1)
if PARTIAL.exists():
    PARTIAL.unlink()
display(adv_df.head())
summary

valid    7876
Name: count, dtype: int64


,orig_URL,URL,label,p_clean,p_adv,n_edits,flipped,duplicate
0,liuy-9a930.web.app,liuys9auu0.web.app,0,0.999926,0.982623,3,False,False
1,mechinchem-5cb8a.web.app,mechinchemoucbva.web.app,0,0.999979,0.886480,3,False,False
2,fb-restriction-case-97be5.web.app,fu-restriction-case-9abel.web.app,0,1.000000,0.992616,3,False,False
3,metafb-tvz6efk.web.app,metafb-tvuiefy.web.app,0,1.000000,0.999661,3,False,False
4,ggsexpole.web.app,ggueepele.web.app,0,0.996644,0.919446,3,False,False


{'model': 'bilstm',
 'split': 'test',
 'phishing': 9694,
 'missed_by_model': 1818,
 'attacked': 7876,
 'flipped': 740,
 'edited_not_flipped': 6392,
 'not_edited': 744,
 'duplicates_dropped': 685,
 'success_within_edits': {1: 0.0085068562722194,
  2: 0.05281868969019807,
  3: 0.09395632300660234},
 'edited_parts': {'subdomain': 5626,
  'fragment': 474,
  'path+fragment': 306,
  'path': 172,
  'subdomain+path': 139,
  'query': 122,
  'path+query': 106,
  'subdomain+query': 63,
  'subdomain+path+fragment': 59,
  'subdomain+fragment': 30,
  'path+query+fragment': 17,
  'subdomain+path+query': 12,
  'subdomain+path+query+fragment': 5,
  'subdomain+query+fragment': 1},
 'flipped_parts': {'fragment': 398,
  'path+fragment': 127,
  'subdomain': 80,
  'query': 59,
  'path+query': 46,
  'path': 29,
  'path+query+fragment': 1},
 'n_edits': {0: 744, 1: 115, 2: 402, 3: 6615}}

## Evaluate Original Plus Adversarial Test Rows

In [ ]:
edited = attack_impl.adversarial_rows(adv_df)
test = splits["test"]
p_test = p_phish(test.URL.tolist())
both = pd.concat([test[["URL", "label"]], edited[["URL", "label"]]], ignore_index=True)
p_both = np.concatenate([p_test, edited.p_adv.values])
m_orig = attack_impl.evaluate(test.label, p_test)
m_adv = attack_impl.evaluate(both.label, p_both)
metrics = attack_impl.adversarial_metrics(adv_df.label, adv_df.p_clean, adv_df.p_adv)
metrics["adversarial_recall"] = attack_impl.adversarial_recall(edited.p_adv)
metrics.update(attack_impl.deltas(m_orig, m_adv))
attack_impl.save_preds(RESULTS_DIR / "bilstm_test_adversarial.csv", both.URL, both.label, p_both)
with (RESULTS_DIR / "bilstm_test_adversarial.json").open("w") as file:
    json.dump({**m_adv, **metrics}, file, indent=1)
display(pd.DataFrame({"test_original": m_orig, "test_original + test_adversarial": m_adv}).drop("confusion_matrix").T.astype(float).round(4))
metrics

,accuracy,precision,recall,f1,roc_auc,pr_auc
test_original,0.9077,0.9607,0.8125,0.8804,0.9445,0.9486
test_original + test_adversarial,0.9028,0.9768,0.8415,0.9041,0.9485,0.9679


{'attack_success_rate': 0.09395632300660234,
 'n_attempts': 7876,
 'confidence_reduction': 0.10315367579460144,
 'adversarial_recall': 0.8852179308205367,
 'delta_accuracy': -0.004884150872733195,
 'delta_recall': 0.02906027470138306,
 'delta_f1': 0.02375350269611931}

In [ ]:
# Load URLBERT adversarial URLs. Change this path if the CSV is elsewhere.
URLBERT_ADV_PATH = Path("../data/urlbert_adv_test.csv")
import pandas as pd
import numpy as np
from pathlib import Path
# p_phish is already defined in cell 5.

if not URLBERT_ADV_PATH.exists():
    matches = [
        path
        for folder in (Path("../data"), RESULTS_DIR)
        for path in folder.glob("*urlbert*adv*.csv")
    ]
    if not matches:
        raise FileNotFoundError("Could not find a URLBERT adversarial CSV.")
    URLBERT_ADV_PATH = next(
        (path for path in matches if "test" in path.name.lower()), matches[0]
    )

urlbert_raw = pd.read_csv(URLBERT_ADV_PATH)
columns = {column.lower(): column for column in urlbert_raw.columns}

url_col = next(
    (columns[name] for name in ("url", "adv_url", "adversarial_url", "generated_url") if name in columns),
    None,
)
if url_col is None:
    raise ValueError(f"No adversarial URL column found in {list(urlbert_raw.columns)}")

orig_col = next(
    (columns[name] for name in ("orig_url", "original_url", "source_url") if name in columns),
    None,
)
label_col = next(
    (columns[name] for name in ("label", "y") if name in columns),
    None,
)

urlbert_rows = pd.DataFrame({"URL": urlbert_raw[url_col]})
urlbert_rows["label"] = (
    pd.to_numeric(urlbert_raw[label_col], errors="raise").astype(int)
    if label_col
    else 0  # URLBERT adversarial examples are assumed to be phishing.
)
if orig_col:
    urlbert_rows["orig_URL"] = urlbert_raw[orig_col]

urlbert_rows = urlbert_rows.dropna(subset=["URL"]).drop_duplicates("URL").reset_index(drop=True)
urlbert_rows["p_adv"] = p_phish(urlbert_rows.URL.tolist())

# Evaluate URLBERT examples, then add only URLs not already present in the test set.
urlbert_metrics = attack_impl.evaluate(urlbert_rows.label, urlbert_rows.p_adv)
new_rows = urlbert_rows[~urlbert_rows.URL.isin(test.URL)].copy()
new_metrics = attack_impl.evaluate(new_rows.label, new_rows.p_adv)

both_urlbert = pd.concat(
    [test[["URL", "label"]], new_rows[["URL", "label"]]], ignore_index=True
)
p_both_urlbert = np.concatenate([p_test, new_rows.p_adv.to_numpy()])
combined_metrics = attack_impl.evaluate(both_urlbert.label, p_both_urlbert)

print("URLBERT file:", URLBERT_ADV_PATH)
print(f"Unique adversarial URLs: {len(urlbert_rows)} | new URLs added to test: {len(new_rows)}")
display(pd.DataFrame({
    "clean_test": m_orig,
    "URLBERT_adversarial_only": urlbert_metrics,
    "new_URLBERT_URLs_only": new_metrics,
    "test_plus_new_URLBERT": combined_metrics,
}).drop(index="confusion_matrix", errors="ignore").T.astype(float).round(4))

attack_impl.deltas(m_orig, combined_metrics)

NameError: name 'p_phish' is not defined